## Mount Drive + Imports + Settings

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers -q

import os
import re
import random
import unicodedata

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel

In [ ]:
DATA_PATH = "/content/drive/MyDrive/SML/News_Category_Dataset_v3.json"
OUTPUT_DIR = "/content/drive/MyDrive/SML/"

CONTEXT_COLUMNS = ["L1", "L2"]
RUN_TAG = "L1_L2"

TOP_K_CATEGORIES = 10
SAMPLE_PER_CLASS = 2000
RANDOM_STATE = 42

OUTER_FOLDS = 10
INNER_FOLDS = 3

BERT_MODEL = "bert-base-uncased"
BERT_MAX_LENGTH = 80

BERT_PRECOMPUTE_BATCH_SIZE = 32

SAVE_EMBEDDING_DTYPE = np.float16

EPOCHS = 1
LEARNING_RATE = 0.001
DROPOUT_RATE = 0.3

LSTM_BATCH_SIZE = 128

HIDDEN_DIM_VALUES = [128, 256, 384]

## Load Data + Build L1-L4

In [ ]:
data = pd.read_json(DATA_PATH, lines=True)

data = data[["category", "headline", "short_description"]]

data = data.dropna()

data["category"] = data["category"].astype(str).str.strip()
data["headline"] = data["headline"].astype(str).str.strip()
data["short_description"] = data["short_description"].astype(str).str.strip()

data = data[
    (data["category"] != "") &
    (data["headline"] != "") &
    (data["short_description"] != "")
].copy()

top_10_categories = data["category"].value_counts().head(TOP_K_CATEGORIES).index.tolist()

data = data[data["category"].isin(top_10_categories)].copy()

sampled_data = []

for category in top_10_categories:
    category_data = data[data["category"] == category]

    category_sample = category_data.sample(
        n=SAMPLE_PER_CLASS,
        random_state=RANDOM_STATE
    )

    sampled_data.append(category_sample)

data = pd.concat(sampled_data)

data = data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

category_to_label = {}

for i in range(len(top_10_categories)):
    category_to_label[top_10_categories[i]] = i

labels = []

for category in data["category"]:
    labels.append(category_to_label[category])

data["label"] = labels

In [ ]:
def get_first_tokens(text, number_of_tokens):
    tokens = str(text).split()
    return " ".join(tokens[:number_of_tokens])


data["L1"] = data["headline"].apply(lambda x: get_first_tokens(x, 5))

data["L2"] = data["headline"]

data["L3"] = data["headline"] + " " + data["short_description"].apply(
    lambda x: get_first_tokens(x, 15)
)

data["L4"] = data["headline"] + " " + data["short_description"]

data = data[
    [
        "category",
        "label",
        "headline",
        "short_description",
        "L1",
        "L2",
        "L3",
        "L4"
    ]
].copy()

data.to_csv(OUTPUT_DIR + "processed_news.csv", index=False)

label_rows = []

for category in top_10_categories:
    label_rows.append({
        "category": category,
        "label": category_to_label[category]
    })

label_mapping = pd.DataFrame(label_rows)

label_mapping.to_csv(OUTPUT_DIR + "label_mapping.csv", index=False)

print("processed_news.csv saved")
print("label_mapping.csv saved")
print("Data shape:", data.shape)
print(data["category"].value_counts())
print(label_mapping)

## CV + Metrics

In [ ]:
def make_stratified_folds(y, number_of_folds, random_state):
    y = np.array(y)

    rng = np.random.default_rng(random_state)

    folds = []

    for i in range(number_of_folds):
        folds.append([])

    unique_labels = np.unique(y)

    for label in unique_labels:
        label_indices = np.where(y == label)[0]
        rng.shuffle(label_indices)

        split_indices = np.array_split(label_indices, number_of_folds)

        for fold_number in range(number_of_folds):
            folds[fold_number].extend(split_indices[fold_number].tolist())

    final_folds = []

    for fold in folds:
        fold = np.array(fold)
        rng.shuffle(fold)
        final_folds.append(fold)

    return final_folds


def calculate_accuracy(y_true, y_pred):
    correct_count = 0

    for i in range(len(y_true)):
        if y_true[i] == y_pred[i]:
            correct_count += 1

    return correct_count / len(y_true)


def calculate_macro_f1(y_true, y_pred):
    labels = np.unique(y_true)

    f1_scores = []

    for label in labels:
        true_positive = 0
        false_positive = 0
        false_negative = 0

        for i in range(len(y_true)):
            if y_true[i] == label and y_pred[i] == label:
                true_positive += 1
            elif y_true[i] != label and y_pred[i] == label:
                false_positive += 1
            elif y_true[i] == label and y_pred[i] != label:
                false_negative += 1

        if true_positive + false_positive == 0:
            precision = 0
        else:
            precision = true_positive / (true_positive + false_positive)

        if true_positive + false_negative == 0:
            recall = 0
        else:
            recall = true_positive / (true_positive + false_negative)

        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        f1_scores.append(f1)

    return np.mean(f1_scores)


def calculate_weighted_f1(y_true, y_pred):
    labels = np.unique(y_true)

    total_count = len(y_true)
    weighted_sum = 0

    for label in labels:
        true_positive = 0
        false_positive = 0
        false_negative = 0
        support = 0

        for i in range(len(y_true)):
            if y_true[i] == label:
                support += 1

            if y_true[i] == label and y_pred[i] == label:
                true_positive += 1
            elif y_true[i] != label and y_pred[i] == label:
                false_positive += 1
            elif y_true[i] == label and y_pred[i] != label:
                false_negative += 1

        if true_positive + false_positive == 0:
            precision = 0
        else:
            precision = true_positive / (true_positive + false_positive)

        if true_positive + false_negative == 0:
            recall = 0
        else:
            recall = true_positive / (true_positive + false_negative)

        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        weighted_sum += f1 * support

    return weighted_sum / total_count


def calculate_per_class_f1(y_true, y_pred):
    labels = np.unique(y_true)

    result = {}

    for label in labels:
        true_positive = 0
        false_positive = 0
        false_negative = 0

        for i in range(len(y_true)):
            if y_true[i] == label and y_pred[i] == label:
                true_positive += 1
            elif y_true[i] != label and y_pred[i] == label:
                false_positive += 1
            elif y_true[i] == label and y_pred[i] != label:
                false_negative += 1

        if true_positive + false_positive == 0:
            precision = 0
        else:
            precision = true_positive / (true_positive + false_positive)

        if true_positive + false_negative == 0:
            recall = 0
        else:
            recall = true_positive / (true_positive + false_negative)

        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        result[int(label)] = f1

    return result

## Seed Function

In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## Precompute Frozen BERT Token Embeddings

In [ ]:
def precompute_bert_token_embeddings_for_one_context(
    text_list,
    context_column,
    model_name,
    max_length,
    batch_size,
    output_dir
):
    embedding_path = output_dir + f"bert_token_embeddings_{context_column}.npy"
    mask_path = output_dir + f"bert_attention_masks_{context_column}.npy"

    total = len(text_list)

    if os.path.exists(embedding_path) and os.path.exists(mask_path):
        print("Existing BERT token embeddings found. Loading:")
        print(embedding_path)
        print(mask_path)

        embeddings = np.load(embedding_path, mmap_mode="r")
        masks = np.load(mask_path, mmap_mode="r")

        print("Embedding shape:", embeddings.shape, embeddings.dtype)
        print("Mask shape:", masks.shape, masks.dtype)

        return embeddings, masks

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Precompute BERT device:", device)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    bert_model = AutoModel.from_pretrained(model_name)

    bert_model.to(device)
    bert_model.eval()

    embedding_shape = (total, max_length, 768)
    mask_shape = (total, max_length)

    embeddings_mmap = np.lib.format.open_memmap(
        embedding_path,
        mode="w+",
        dtype=SAVE_EMBEDDING_DTYPE,
        shape=embedding_shape
    )

    masks_mmap = np.lib.format.open_memmap(
        mask_path,
        mode="w+",
        dtype=np.int8,
        shape=mask_shape
    )

    for start_index in range(0, total, batch_size):
        end_index = min(start_index + batch_size, total)
        batch_texts = text_list[start_index:end_index]

        encoded = tokenizer(
            list(batch_texts),
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.no_grad():
            outputs = bert_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        token_embeddings = outputs.last_hidden_state

        embeddings_mmap[start_index:end_index] = (
            token_embeddings.cpu().numpy().astype(SAVE_EMBEDDING_DTYPE)
        )

        masks_mmap[start_index:end_index] = (
            attention_mask.cpu().numpy().astype(np.int8)
        )

        if start_index % (batch_size * 100) == 0:
            print(f"{context_column} BERT progress: {start_index}/{total}")

    embeddings_mmap.flush()
    masks_mmap.flush()

    del bert_model
    del embeddings_mmap
    del masks_mmap
    torch.cuda.empty_cache()

    embeddings = np.load(embedding_path, mmap_mode="r")
    masks = np.load(mask_path, mmap_mode="r")

    print("Saved:", embedding_path)
    print("Saved:", mask_path)
    print("Embedding shape:", embeddings.shape, embeddings.dtype)
    print("Mask shape:", masks.shape, masks.dtype)

    return embeddings, masks

## Dataset for Precomputed Embeddings

In [ ]:
class BertEmbeddingDataset(Dataset):
    def __init__(self, embeddings, masks, labels, indices):
        self.embeddings = embeddings
        self.masks = masks
        self.labels = np.array(labels, dtype=np.int64)
        self.indices = np.array(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        real_index = self.indices[index]

        embedding = torch.tensor(
            self.embeddings[real_index],
            dtype=torch.float32
        )

        mask = torch.tensor(
            self.masks[real_index],
            dtype=torch.long
        )

        label = torch.tensor(
            self.labels[real_index],
            dtype=torch.long
        )

        return embedding, mask, label

## BiLSTM Model on Frozen BERT Embeddings

In [ ]:
class BiLSTMOnFrozenBertEmbeddingsClassifier(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_classes,
        dropout_rate=0.3
    ):
        super(BiLSTMOnFrozenBertEmbeddingsClassifier, self).__init__()

        self.bilstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout_rate)

        self.classifier = nn.Linear(
            hidden_dim * 2,
            num_classes
        )

    def forward(self, token_embeddings, attention_mask):
        lstm_output, _ = self.bilstm(token_embeddings)

        expanded_mask = attention_mask.unsqueeze(-1).float()

        masked_output = lstm_output * expanded_mask

        summed = torch.sum(masked_output, dim=1)

        counts = torch.clamp(
            expanded_mask.sum(dim=1),
            min=1e-9
        )

        pooled = summed / counts

        pooled = self.dropout(pooled)

        logits = self.classifier(pooled)

        return logits

## Train and Predict

In [ ]:
def train_bilstm_on_frozen_bert_embeddings_and_predict(
    bert_embeddings,
    bert_masks,
    labels,
    train_indices,
    test_indices,
    hidden_dim,
    random_state
):
    set_all_seeds(random_state)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    train_dataset = BertEmbeddingDataset(
        embeddings=bert_embeddings,
        masks=bert_masks,
        labels=labels,
        indices=train_indices
    )

    test_dataset = BertEmbeddingDataset(
        embeddings=bert_embeddings,
        masks=bert_masks,
        labels=labels,
        indices=test_indices
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=LSTM_BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=LSTM_BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    num_classes = len(np.unique(labels))

    model = BiLSTMOnFrozenBertEmbeddingsClassifier(
        input_dim=768,
        hidden_dim=hidden_dim,
        num_classes=num_classes,
        dropout_rate=DROPOUT_RATE
    )

    model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    model.train()

    for epoch in range(EPOCHS):
        total_loss = 0

        for batch_embeddings, batch_masks, batch_y in train_loader:
            batch_embeddings = batch_embeddings.to(device)
            batch_masks = batch_masks.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(
                token_embeddings=batch_embeddings,
                attention_mask=batch_masks
            )

            loss = criterion(logits, batch_y)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        average_loss = total_loss / len(train_loader)
        print("Epoch:", epoch + 1, "| loss:", round(average_loss, 4))

    model.eval()

    predictions = []

    with torch.no_grad():
        for batch_embeddings, batch_masks, _ in test_loader:
            batch_embeddings = batch_embeddings.to(device)
            batch_masks = batch_masks.to(device)

            logits = model(
                token_embeddings=batch_embeddings,
                attention_mask=batch_masks
            )

            batch_pred = torch.argmax(logits, dim=1)

            predictions.extend(batch_pred.cpu().numpy().tolist())

    return np.array(predictions)

## Inner CV

In [ ]:
def tune_bilstm_frozen_bert_hidden_dim_with_inner_cv(
    bert_embeddings,
    bert_masks,
    labels,
    outer_train_indices,
    hidden_dim_values,
    inner_folds_number,
    random_state
):
    y_train_outer = labels[outer_train_indices]

    inner_folds = make_stratified_folds(
        y_train_outer,
        inner_folds_number,
        random_state
    )

    all_outer_train_positions = np.arange(len(outer_train_indices))

    best_hidden_dim = None
    best_score = -1

    for hidden_dim in hidden_dim_values:
        fold_scores = []

        print()
        print("Trying hidden_dim =", hidden_dim)

        for inner_fold_number, valid_positions in enumerate(inner_folds, start=1):
            print("Inner fold:", inner_fold_number)

            train_positions = np.setdiff1d(
                all_outer_train_positions,
                valid_positions
            )

            inner_train_indices = outer_train_indices[train_positions]
            inner_valid_indices = outer_train_indices[valid_positions]

            y_inner_valid = labels[inner_valid_indices]

            y_valid_pred = train_bilstm_on_frozen_bert_embeddings_and_predict(
                bert_embeddings=bert_embeddings,
                bert_masks=bert_masks,
                labels=labels,
                train_indices=inner_train_indices,
                test_indices=inner_valid_indices,
                hidden_dim=hidden_dim,
                random_state=random_state + inner_fold_number
            )

            macro_f1 = calculate_macro_f1(
                y_inner_valid,
                y_valid_pred
            )

            fold_scores.append(macro_f1)

        average_score = np.mean(fold_scores)

        print(
            "hidden_dim =",
            hidden_dim,
            "| inner macro-F1 =",
            round(average_score, 4)
        )

        if average_score > best_score:
            best_score = average_score
            best_hidden_dim = hidden_dim

    return best_hidden_dim, best_score

## Outer Nested CV for One Context Level

In [ ]:
def run_nested_cv_bilstm_frozen_bert_one_setting(
    data,
    context_column,
    hidden_dim_values,
    bert_embeddings,
    bert_masks
):
    labels = data["label"].to_numpy()

    outer_folds = make_stratified_folds(
        labels,
        OUTER_FOLDS,
        random_state=RANDOM_STATE
    )

    all_indices = np.arange(len(labels))

    fold_rows = []
    per_class_rows = []

    for outer_fold_number, test_indices in enumerate(outer_folds, start=1):
        print()
        print("====================================")
        print("Model: BiLSTM + precomputed Frozen BERT token embeddings")
        print("Context:", context_column)
        print("Outer fold:", outer_fold_number)
        print("====================================")

        train_indices = np.setdiff1d(all_indices, test_indices)

        best_hidden_dim, best_inner_macro_f1 = tune_bilstm_frozen_bert_hidden_dim_with_inner_cv(
            bert_embeddings=bert_embeddings,
            bert_masks=bert_masks,
            labels=labels,
            outer_train_indices=train_indices,
            hidden_dim_values=hidden_dim_values,
            inner_folds_number=INNER_FOLDS,
            random_state=outer_fold_number
        )

        y_test = labels[test_indices]

        y_test_pred = train_bilstm_on_frozen_bert_embeddings_and_predict(
            bert_embeddings=bert_embeddings,
            bert_masks=bert_masks,
            labels=labels,
            train_indices=train_indices,
            test_indices=test_indices,
            hidden_dim=best_hidden_dim,
            random_state=RANDOM_STATE + outer_fold_number
        )

        test_accuracy = calculate_accuracy(y_test, y_test_pred)
        test_macro_f1 = calculate_macro_f1(y_test, y_test_pred)
        test_weighted_f1 = calculate_weighted_f1(y_test, y_test_pred)

        print("Best hidden_dim:", best_hidden_dim)
        print("Test accuracy:", test_accuracy)
        print("Test macro-F1:", test_macro_f1)
        print("Test weighted-F1:", test_weighted_f1)

        fold_rows.append({
            "context_level": context_column,
            "algorithm": "BiLSTM",
            "representation": "precomputed_frozen_bert_token_embeddings",
            "outer_fold": outer_fold_number,
            "best_hyperparameter_name": "hidden_dim",
            "best_hyperparameter": best_hidden_dim,
            "inner_macro_f1": best_inner_macro_f1,
            "test_accuracy": test_accuracy,
            "test_macro_f1": test_macro_f1,
            "test_weighted_f1": test_weighted_f1
        })

        per_class_f1 = calculate_per_class_f1(y_test, y_test_pred)

        for label in per_class_f1:
            per_class_rows.append({
                "context_level": context_column,
                "algorithm": "BiLSTM",
                "representation": "precomputed_frozen_bert_token_embeddings",
                "outer_fold": outer_fold_number,
                "label": label,
                "f1": per_class_f1[label]
            })

        # Save progress after every outer fold
        temp_fold_results = pd.DataFrame(fold_rows)
        temp_per_class_results = pd.DataFrame(per_class_rows)

        temp_fold_results.to_csv(
            OUTPUT_DIR + f"bilstm_precomputed_frozen_bert_fold_results_{context_column}_progress.csv",
            index=False
        )

        temp_per_class_results.to_csv(
            OUTPUT_DIR + f"bilstm_precomputed_frozen_bert_per_class_f1_results_{context_column}_progress.csv",
            index=False
        )

    fold_results = pd.DataFrame(fold_rows)
    per_class_results = pd.DataFrame(per_class_rows)

    return fold_results, per_class_results

## Run Selected Levels and Save

In [ ]:
print("Running context levels:", CONTEXT_COLUMNS)
print("Run tag:", RUN_TAG)
print("Sample per class:", SAMPLE_PER_CLASS)
print("Outer folds:", OUTER_FOLDS)
print("Inner folds:", INNER_FOLDS)
print("Hidden dim values:", HIDDEN_DIM_VALUES)
print("Epochs:", EPOCHS)
print("LSTM batch size:", LSTM_BATCH_SIZE)
print("BERT max length:", BERT_MAX_LENGTH)

In [ ]:
all_frozen_bert_bilstm_fold_results = []
all_frozen_bert_bilstm_per_class_results = []

for context_column in CONTEXT_COLUMNS:
    print("=" * 50)
    print("Preparing BERT token embeddings for:", context_column)
    print("=" * 50)

    text_list = data[context_column].astype(str).tolist()

    bert_embeddings, bert_masks = precompute_bert_token_embeddings_for_one_context(
        text_list=text_list,
        context_column=context_column,
        model_name=BERT_MODEL,
        max_length=BERT_MAX_LENGTH,
        batch_size=BERT_PRECOMPUTE_BATCH_SIZE,
        output_dir=OUTPUT_DIR
    )

    fold_results, per_class_results = run_nested_cv_bilstm_frozen_bert_one_setting(
        data=data,
        context_column=context_column,
        hidden_dim_values=HIDDEN_DIM_VALUES,
        bert_embeddings=bert_embeddings,
        bert_masks=bert_masks
    )

    fold_results.to_csv(
        OUTPUT_DIR + f"bilstm_precomputed_frozen_bert_fold_results_{context_column}.csv",
        index=False
    )

    per_class_results.to_csv(
        OUTPUT_DIR + f"bilstm_precomputed_frozen_bert_per_class_f1_results_{context_column}.csv",
        index=False
    )

    all_frozen_bert_bilstm_fold_results.append(fold_results)
    all_frozen_bert_bilstm_per_class_results.append(per_class_results)

    print(context_column, "saved to Google Drive.")

## Save Summary for This Notebook

In [ ]:
frozen_bert_bilstm_fold_results_partial = pd.concat(
    all_frozen_bert_bilstm_fold_results,
    ignore_index=True
)

frozen_bert_bilstm_per_class_results_partial = pd.concat(
    all_frozen_bert_bilstm_per_class_results,
    ignore_index=True
)

frozen_bert_bilstm_fold_results_partial.to_csv(
    OUTPUT_DIR + f"bilstm_precomputed_frozen_bert_fold_results_{RUN_TAG}.csv",
    index=False
)

frozen_bert_bilstm_per_class_results_partial.to_csv(
    OUTPUT_DIR + f"bilstm_precomputed_frozen_bert_per_class_f1_results_{RUN_TAG}.csv",
    index=False
)

summary_rows = []

for context_column in CONTEXT_COLUMNS:
    context_result = frozen_bert_bilstm_fold_results_partial[
        frozen_bert_bilstm_fold_results_partial["context_level"] == context_column
    ]

    summary_rows.append({
        "context_level": context_column,
        "algorithm": "BiLSTM",
        "representation": "precomputed_frozen_bert_token_embeddings",
        "mean_accuracy": context_result["test_accuracy"].mean(),
        "std_accuracy": context_result["test_accuracy"].std(),
        "mean_macro_f1": context_result["test_macro_f1"].mean(),
        "std_macro_f1": context_result["test_macro_f1"].std(),
        "mean_weighted_f1": context_result["test_weighted_f1"].mean(),
        "std_weighted_f1": context_result["test_weighted_f1"].std()
    })

frozen_bert_bilstm_summary_partial = pd.DataFrame(summary_rows)

frozen_bert_bilstm_summary_partial.to_csv(
    OUTPUT_DIR + f"bilstm_precomputed_frozen_bert_summary_results_{RUN_TAG}.csv",
    index=False
)

print("Partial summary saved.")
print(frozen_bert_bilstm_summary_partial)